In [1]:
library(data.table)

# ============================================================
# EFP ANOMALY MEMORY VARIABLES
# Compute site-level z-score anomalies of GPPsat, NEPmax,
# ETmax, uWUE from the EFP table, then create lag-1 and lag-2
# columns to use as ecological memory predictors.
# ============================================================

efp_file   <- "fluxnet_2017_2025_V02/EFP_outputs_corrected/ALL_SITES_YEARLY_EFP_complete4continuousYears.csv"
model_file <- "derived_tables/outputs_afterEGU_results/EFP_mortality_trait_hydro_combined.csv"
out_dir    <- "derived_tables/outputs_afterEGU_results"

# minimum years per site to compute a stable baseline
MIN_YEARS <- 4

EFP_VARS  <- c("GPPsat", "NEPmax", "ETmax", "uWUE")

In [2]:
# ------------------------------------------------------------
# 1) Load data
# ------------------------------------------------------------

efp_dt   <- fread(efp_file)
model_dt <- fread(model_file)

# harmonize year column to lowercase in efp_dt
if ("YEAR" %in% names(efp_dt))  setnames(efp_dt, "YEAR", "year")
if ("year" %in% names(model_dt)) setnames(model_dt, "year", "YEAR")

cat("EFP table:   ", nrow(efp_dt),   "rows,", uniqueN(efp_dt$SITE_ID),   "sites\n")
cat("Model table: ", nrow(model_dt), "rows,", uniqueN(model_dt$SITE_ID), "sites\n")
cat("Model years: ", min(model_dt$YEAR), "-", max(model_dt$YEAR), "\n")

EFP table:    1180 rows, 184 sites
Model table:  1180 rows, 184 sites
Model years:  2017 - 2025 


In [3]:
# ------------------------------------------------------------
# 2) Compute site-level baseline (mean + SD) across all years
# ------------------------------------------------------------

# keep only sites in the modelling dataset for anomaly calculation
model_sites <- unique(model_dt$SITE_ID)
efp_sub     <- efp_dt[SITE_ID %in% model_sites]

# flag sites with fewer than MIN_YEARS observations
site_nyears <- efp_sub[, .(n_years = .N), by = SITE_ID]
low_data_sites <- site_nyears[n_years < MIN_YEARS, SITE_ID]

if (length(low_data_sites) > 0) {
  cat("Sites with fewer than", MIN_YEARS, "years of EFP data (anomalies will be NA):\n")
  cat(paste(low_data_sites, collapse = ", "), "\n")
}

# compute site-level mean and SD per EFP variable
baseline_list <- lapply(EFP_VARS, function(v) {
  efp_sub[, .(
    mean_val = mean(get(v), na.rm = TRUE),
    sd_val   = sd(get(v),   na.rm = TRUE),
    n_years  = sum(!is.na(get(v)))
  ), by = SITE_ID][, variable := v]
})
baseline_long <- rbindlist(baseline_list)
# zero SD (constant site) -> NA to avoid Inf anomalies
baseline_long[sd_val == 0, sd_val := NA_real_]
# sites below MIN_YEARS get NA baseline so anomaly stays NA
baseline_long[n_years < MIN_YEARS, c("mean_val", "sd_val") := NA_real_]

cat("Baseline computed for", uniqueN(baseline_long$SITE_ID), "sites\n")

Baseline computed for 184 sites


In [4]:
# ------------------------------------------------------------
# 3) Compute z-score anomalies for each site-year
# ------------------------------------------------------------

anom_list <- lapply(EFP_VARS, function(v) {
  bl <- baseline_long[variable == v, .(SITE_ID, mean_val, sd_val)]
  dt <- merge(efp_sub[, .(SITE_ID, year, val = get(v))], bl, by = "SITE_ID", all.x = TRUE)
  dt[, anom := (val - mean_val) / sd_val]
  dt[, .(SITE_ID, year, anom)]
})
names(anom_list) <- EFP_VARS

# combine into one wide table
anom_dt <- Reduce(function(a, b) merge(a, b, by = c("SITE_ID", "year"), all = TRUE),
                  Map(function(dt, nm) setnames(dt, "anom", paste0(nm, "_anom")),
                      anom_list, EFP_VARS))

cat("Anomaly table:", nrow(anom_dt), "rows\n")
print(head(anom_dt))

Anomaly table: 1180 rows
Key: <SITE_ID, year>
   SITE_ID  year GPPsat_anom NEPmax_anom   ETmax_anom  uWUE_anom
    <char> <int>       <num>       <num>        <num>      <num>
1:  AT-Mmg  2021  -0.9296875   1.1446783 -0.984495345 -1.3308971
2:  AT-Mmg  2022  -0.6458101  -1.0107719 -0.482699218  0.5335418
3:  AT-Mmg  2023   0.2991211   0.5071052  0.132997399 -0.1608571
4:  AT-Mmg  2024   1.2763765  -0.6410116  1.334197164  0.9582124
5:  AT-Nsd  2018  -1.3385079  -1.2309922 -1.437137590 -1.3533189
6:  AT-Nsd  2019  -0.5756271  -0.6402265  0.008666874 -0.4630085


In [5]:
# ------------------------------------------------------------
# 4) Create lag-1 and lag-2 anomaly columns
# Each column = anomaly from (year - lag) for that SITE_ID
# ------------------------------------------------------------

setkey(anom_dt, SITE_ID, year)

anom_cols <- paste0(EFP_VARS, "_anom")

lag_dt <- copy(anom_dt)

for (lag in 1:2) {
  # create a shifted version: year + lag joins to the target year
  shifted <- copy(anom_dt)
  shifted[, year := year + lag]  # shift so YEAR=X carries values from YEAR=X-lag
  setnames(shifted, anom_cols, paste0(anom_cols, "_lag", lag))
  lag_dt <- merge(lag_dt, shifted, by = c("SITE_ID", "year"), all.x = TRUE)
}

cat("Lag table columns:\n")
cat(paste(names(lag_dt), collapse = ", "), "\n")
cat("Rows:", nrow(lag_dt), "\n")

Lag table columns:
SITE_ID, year, GPPsat_anom, NEPmax_anom, ETmax_anom, uWUE_anom, GPPsat_anom_lag1, NEPmax_anom_lag1, ETmax_anom_lag1, uWUE_anom_lag1, GPPsat_anom_lag2, NEPmax_anom_lag2, ETmax_anom_lag2, uWUE_anom_lag2 
Rows: 1180 


In [6]:
# ------------------------------------------------------------
# 5) Check coverage before merging
# ------------------------------------------------------------

lag1_cols <- paste0(anom_cols, "_lag1")

# how many model site-years will have lag-1 coverage?
lag_check <- merge(
  model_dt[, .(SITE_ID, YEAR)],
  lag_dt[, c("SITE_ID", "year", lag1_cols), with = FALSE],
  by.x = c("SITE_ID", "YEAR"),
  by.y = c("SITE_ID", "year"),
  all.x = TRUE
)

for (col in lag1_cols) {
  n_available <- sum(!is.na(lag_check[[col]]))
  pct <- round(100 * n_available / nrow(lag_check), 1)
  cat(col, ": ", n_available, "/", nrow(lag_check), "(", pct, "%) available\n", sep = "")
}

GPPsat_anom_lag1: 983/1180(83.3%) available
NEPmax_anom_lag1: 983/1180(83.3%) available
ETmax_anom_lag1: 983/1180(83.3%) available
uWUE_anom_lag1: 983/1180(83.3%) available


In [7]:
# ------------------------------------------------------------
# 6) Merge into model dataset and save
# ------------------------------------------------------------

# columns to add: lag-1 and lag-2 anomalies only (not current-year anom,
# which would be partly collinear with absolute EFPs already in model)
lag_cols_all <- c(paste0(anom_cols, "_lag1"), paste0(anom_cols, "_lag2"))

model_dt_out <- merge(
  model_dt,
  lag_dt[, c("SITE_ID", "year", lag_cols_all), with = FALSE],
  by.x = c("SITE_ID", "YEAR"),
  by.y = c("SITE_ID", "year"),
  all.x = TRUE
)

out_file <- file.path(out_dir, "EFP_mortality_trait_hydro_combined_with_EFPanom_memory.csv")
fwrite(model_dt_out, out_file)

cat("Saved:", out_file, "\n")
cat("Rows:", nrow(model_dt_out), "\n")
cat("New columns added:\n")
cat(paste(lag_cols_all, collapse = "\n"), "\n")

Saved: derived_tables/outputs_afterEGU_results/EFP_mortality_trait_hydro_combined_with_EFPanom_memory.csv 
Rows: 1180 
New columns added:
GPPsat_anom_lag1
NEPmax_anom_lag1
ETmax_anom_lag1
uWUE_anom_lag1
GPPsat_anom_lag2
NEPmax_anom_lag2
ETmax_anom_lag2
uWUE_anom_lag2 


In [8]:
# ------------------------------------------------------------
# 7) Quick summary of added memory variables
# ------------------------------------------------------------

cat("Summary of lag-1 anomaly variables:\n")
for (col in paste0(anom_cols, "_lag1")) {
  x <- model_dt_out[[col]]
  cat(sprintf("  %-25s  n=%d  mean=%.3f  sd=%.3f  range=[%.2f, %.2f]\n",
              col, sum(!is.na(x)), mean(x, na.rm=TRUE), sd(x, na.rm=TRUE),
              min(x, na.rm=TRUE), max(x, na.rm=TRUE)))
}

Summary of lag-1 anomaly variables:
  GPPsat_anom_lag1           n=983  mean=-0.025  sd=0.909  range=[-2.43, 2.18]
  NEPmax_anom_lag1           n=983  mean=-0.037  sd=0.904  range=[-2.38, 2.42]
  ETmax_anom_lag1            n=983  mean=-0.025  sd=0.909  range=[-2.16, 2.22]
  uWUE_anom_lag1             n=983  mean=-0.008  sd=0.913  range=[-2.35, 2.19]
